# Hide observations while preserving a fair prediction task

This notebook makes one fresh Ambient pose call and constructs the exact masking policies on real cached poses. It connects input construction with the fitted, controlled study.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/v2_data.py').is_file()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/v2_data.py').is_file())
sys.path.insert(0, str(PROJECT))
REPOSITORY = PROJECT.parents[1]
from worldmodels.v2_data import load_cohort
from worldmodels import v2_evaluation as ev
from worldmodels import v2_models as models
from worldmodels.v2_runner import window_errors, verify_freeze
COHORT = PROJECT/'artifacts/runs/reference-v2-cohort-v1'
RUN = PROJECT/'artifacts/runs/reference-v2-core-v1'
TRANSFER = PROJECT/'artifacts/runs/reference-v2-transfer-v1'
FIGURES = PROJECT/'artifacts/executed-reference-v2/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
USED_INPUTS={}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def record_input(path):
    path=Path(path).resolve()
    key=str(path.relative_to(REPOSITORY)) if path.is_relative_to(REPOSITORY) else str(path)
    USED_INPUTS[key]=sha(path)
def read_json(path):
    path = Path(path)
    assert path.is_file(), f'Required completed artifact is absent: {path}. Read docs/reference_v2/README.md for prerequisites.'
    record_input(path)
    return json.loads(path.read_text())
def read_csv(path, **kwargs):
    record_input(path)
    return pd.read_csv(path, **kwargs)
def artifact(name, directory=RUN):
    path = directory/name
    manifest = read_json(directory/'artifact_hashes.json')
    assert name in manifest and path.is_file(), f'Unrecorded or missing artifact: {path}'
    assert sha(path) == manifest[name], f'Artifact hash differs from the completed run: {path}'
    record_input(path)
    return path
def completed_run():
    result = read_json(artifact('results.json'))
    assert result['status']=='complete' and not result.get('test_only', False)
    return result
def save_figure(figure, name):
    figure.savefig(FIGURES/(name+'.svg'), bbox_inches='tight')
    figure.savefig(FIGURES/(name+'.png'), dpi=150, bbox_inches='tight')
freeze = verify_freeze()
record_input(PROJECT/'docs/reference_v2/protocol_freeze.json')
x, valid, metadata, actual_times = load_cohort(COHORT, verify=True)
cohort_identity=read_json(COHORT/'identity.json');record_input(COHORT/'cohort.csv')
for name in cohort_identity['prepared_inputs']:
    record_input(REPOSITORY/cohort_identity['prepared_directory']/name)
assert x.shape == (len(metadata),24,16,2) and len(metadata)==2815
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False})
print('Verified real observations:',len(metadata),'windows;',metadata.source_id.nunique(),'source videos.')
print('Observation units: pelvis-relative 2D coordinates divided by a prefix torso scale. Times are decoder seconds.')


## Decode one real source frame through Ambient

This is a fresh image inference call using the pinned local pose weights. The large study reuses verified full-span caches. GAVD frame numbers are one-based and the OpenCV decoder index is zero-based; the conversion is made once before calling the RGB image API.

In [ ]:
import cv2
from ambient.pose.keypoint_extractor import SequenceKeypointExtractor
from worldmodels.data import JOINT_IDS
row=metadata.sort_values(['source_id','sequence_id','first_frame']).iloc[0]
media=read_csv(COHORT/'source_cache_manifest.csv')
video=REPOSITORY/media.set_index('source_id').loc[row.source_id,'video_path']
weights=REPOSITORY/'data/models/pose_landmarker_lite.task'
assert sha(weights) in set(media.pose_model_sha256)
assert sha(video)==media.set_index('source_id').loc[row.source_id,'video_sha256']
record_input(weights);record_input(video)
capture=cv2.VideoCapture(str(video));capture.set(cv2.CAP_PROP_POS_FRAMES,int(row.first_frame)-1)
ok,frame=capture.read();capture.release();assert ok,'The source frame could not be decoded.'
extractor=SequenceKeypointExtractor()
try:
    pose=extractor.extract_from_image(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB),str(weights))
finally:
    extractor.cleanup()
fig,ax=plt.subplots(figsize=(8,4.5));ax.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
if pose is not None:
    points=[point for point in pose.keypoints if point.id in JOINT_IDS and point.confidence>=.5]
    ax.scatter([p.x for p in points],[p.y for p in points],color='#00b0a0',s=25)
ax.set_title('One actual source frame and the newly estimated observed joints.');ax.axis('off')
plt.show()
print('Fresh pose calls in this teaching cell: 1. This is not a new full-cohort extraction.')


## Build equal-budget corruption policies

A token contains the two coordinates of one joint at one time. We hide only naturally observed tokens. Every observed shoulder and hip used for normalization remains visible, and every joint retains its latest anchor. The four principal corruption policies and their controls hide the same rounded fraction of eligible tokens.

In [ ]:
from worldmodels.v2_masks import POLICIES,MaskConfig,make_mask
index=int(metadata.index[metadata.window_id.eq(row.window_id)][0])
prefix,observed,t=x[index,:16],valid[index,:16],actual_times[index,:16]
designs={policy:make_mask(prefix,observed,t,MaskConfig(policy),seed=27) for policy in POLICIES}
joint_names=dict(zip(JOINT_IDS,['L shoulder','R shoulder','L elbow','R elbow','L wrist','R wrist','L hip','R hip','L knee','R knee','L ankle','R ankle','L heel','R heel','L foot index','R foot index']))
display(pd.DataFrame({'BlazePose ID':JOINT_IDS,'joint':[joint_names[j] for j in JOINT_IDS]}))
display(pd.DataFrame([entry[2] for entry in designs.values()]))
counts=[int(value[0].sum()) for policy,value in designs.items() if policy!='none']
assert len(set(counts))==1
print('M0 is the uncorrupted anchor and deliberately retains more training information.')


## Inspect what is visible and what is predicted

The colored grid distinguishes natural missingness, deliberate corruption, protected observations, and fixed future queries. Purple query locations request the same distal coordinates for every method. Their existence does not depend on whether those future joints will later be observed. A block or tube may be truncated to match the exact budget.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
policies=['uniform','time_blocks','anatomical_tubes','motion_balanced']
colors=['#dce1e5','#c6e5df','#eea04d','#468dbc','#9d89c8','#ffffff']
fig,axes=plt.subplots(2,2,figsize=(13,8),sharex=True,sharey=True)
for ax,policy in zip(axes.flat,policies):
    hidden,protected,audit=designs[policy]
    panel=np.full((24,16),5,int);panel[:16]=observed.astype(int)
    panel[:16][hidden]=2;panel[:16][protected]=3;panel[16:,models.TARGET_JOINTS]=4
    ax.imshow(panel.T,cmap=ListedColormap(colors),vmin=0,vmax=5,aspect='auto',interpolation='nearest')
    ax.axvline(15.5,color='#17394c',lw=1)
    ax.set(title=policy.replace('_',' ').capitalize()+f": {audit['hidden_tokens']}/{audit['eligible_tokens']} eligible tokens hidden",xlabel='Sample index; future begins at 16')
    ax.set_yticks(range(16),[f'{joint_names[j]} ({j})' for j in JOINT_IDS],fontsize=8)
labels=['Naturally missing','Visible prefix','Artificially hidden','Protected context','Fixed future query','Future non-query']
fig.legend(handles=[Patch(facecolor=c,edgecolor='#777777',label=l) for c,l in zip(colors,labels)],loc='lower center',ncol=3,frameon=False)
fig.tight_layout(rect=[0,.1,1,1]);save_figure(fig,'22_matched_mask_policies');plt.show()


## Verify a hidden observation cannot survive in filled features

Coordinates are erased before causal filling. For a fixed mask, changing the hidden coordinates cannot alter the visible feature vector or anchors. This checks the input boundary. It does not eliminate information in an adaptive mask pattern, because the policy observed movement before deciding what to hide.

In [ ]:
hidden,protected,_=designs['motion_balanced']
a,anchor,known=models.causal_features(prefix[None],observed[None],t[None],hidden=hidden[None])
changed=prefix.copy();changed[hidden|~observed]=1e20
b,other_anchor,other_known=models.causal_features(changed[None],observed[None],t[None],hidden=hidden[None])
np.testing.assert_array_equal(a,b);np.testing.assert_array_equal(anchor,other_anchor)
np.testing.assert_array_equal(known,other_known)
assert not (hidden&protected).any() and not (hidden&~observed).any()
print('Fixed-mask hidden-value invariance passed; natural missingness never became a training target.')


## Connect the training task to the fitted experiment

Every mask uses the same backbone, future objective, normal training groups, optimizer settings, and selected number of updates. M3 is compared with random joint groups of the same sizes and time extent. M4 is compared with a motion-focused policy and a pattern-only residual diagnostic. The complete results, including unsuccessful policies, remain visible.

In [ ]:
result=completed_run()
forecast=pd.DataFrame(result['normal_forecast'])
display(forecast[forecast.arm.str.startswith('mlp_') & forecast.horizon.eq(8)][['arm','mse','ci95','normal_sources']])
principal=forecast.query('horizon==8').set_index('arm')
print('The uniform, anatomical-tube, and motion-balanced models have normal-source MSE values of',
      ', '.join(f"{principal.loc[name,'mse']:.5f}" for name in ['mlp_uniform','mlp_anatomical_tubes','mlp_motion_balanced']), 'respectively.')
print('These point estimates must be read with the paired controls, rather than interpreted as an anatomical advantage by design.')
selection=read_json(artifact('selection.json'))
durations=[{'training_sources':len(v['training_sources']),'selected_steps':v['selected_steps']} for v in selection.values() if 'selected_steps' in v]
display(pd.DataFrame(durations))
print('An anatomical masking gain requires its matched controls. A low latent loss is not the endpoint in this coordinate-only study.')

# Retain the exact inputs used by this independent-kernel execution.
input_path=PROJECT/'artifacts/executed-reference-v2/inputs/22_structured_masking_laboratory.json'
input_path.parent.mkdir(parents=True,exist_ok=True)
input_path.write_text(json.dumps({'path_base':'repository root','inputs':USED_INPUTS},indent=2)+'\n')
